In [ ]:
"""
Created on Frid Jan 21 10:02:01 2022
@author: Oumbeg
"""
from bs4 import BeautifulSoup
import datetime
import pandas as pd
import numpy as np
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os
import re
import requests
import tabula
import camelot

print("Running UG BUG Web Scraping Tool v.1.0")


os.environ["PATH"] += r"C:\Program Files (x86)\gs\gs9.54.0\bin"
os.environ["PATH"] += r"C:\Program Files (x86)\gs\gs9.54.0\lib"

#Assigning current time, output file name and ExcelWriter object
now = datetime.datetime.now()
filename = 'UG BUG SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])


#Assigning the folders that are going to be used in the process
scriptfolder = os.path.dirname(os.path.abspath(__file__))
tempfolder = os.path.join(scriptfolder,'tempfolder')
os.chdir(scriptfolder)

#Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
	for temp_file in os.listdir(tempfolder):
		os.remove(os.path.join(tempfolder, temp_file))
else:
	os.mkdir(tempfolder)

regdict={'UG BUG 1': 'https://www.bou.or.ug/bou/bouwebsite/Supervision/supervisedinstitutions.html'}

chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory" : tempfolder, 
        "plugins.always_open_pdf_externally": True}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

# Defined Functions

def findUrl(mydata):
    regex = re.compile(r"(?i)\b((?:https?://|www\d{2,4,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:'\".,<>?«»“”‘’]))")
    url = regex.findall(mydata)
    return ' '.join([str(elem) for elem in [x[0] for x in url]])

    
def findEmail(myData):
    """
    This function finds email in a string.
    :param myData: string
    :return: String
    """
    regex = re.compile('[a-zA-Z0-9.\-_]+@[^@]+\.[^@]+')
    email = regex.findall(myData)
    return ' '.join([str(elem) for elem in email])

# try to create an empty folder "tempfolder"
try:
    os.mkdir(tempfolder)
except:
    prevfiles=os.listdir(tempfolder)
    for prf in prevfiles:
        os.remove(os.path.join(tempfolder, prf))

processdate=now.strftime('%Y-%m-%d')

pattern = re.compile('([0-9]+)')

for reg in regdict:
    
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(3)
    
    # the ListCode
    listCode = 0
    
    # the block where to find the links to download the PDFs 
    soup = BeautifulSoup(driver.page_source, "html.parser")
    table = soup.find("div",{"class":"contenttext"})
    typology = table.find_all("li")
    
    
    for m in range(len(typology)):
        d = m+1
        
        if d <= 5 and d not in [3,4]:
            driver.find_element(By.XPATH, '//*[@id="apollo-page"]/section/div/div[2]/div[2]/div[1]/div/div/div[2]/div/div/div/ul[1]/li['+ str(d) +']/a').click()
            while len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:
                print('Waiting for file to download')
                sleep(2)
    
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            print(m,'---',typology[m].text)
            
            tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')
            # Iterate over the tables of each pages
            for i in range(tables.n):
                df_Table = tables[i].df
        
                for j in range(len(df_Table)):
                    if df_Table[1][j].strip() not in ['','NAME'] :
                        #print('Name: ',' '.join([item.strip() for item in df_Table[1][j].splitlines()]).strip())
                        #print('Address: ',' '.join([item.strip() for item in df_Table[2][j].splitlines()]).strip())
                        #print('TEL: ',' '.join([item.strip() for item in df_Table[3][j].splitlines()]).strip())
                        #print('FAX: ',' '.join([item.strip() for item in df_Table[4][j].splitlines()]).strip(),'\n')
                        sqldict['Name'].append(' '.join([item.strip() for item in df_Table[1][j].splitlines()]).strip())
                        sqldict['Address_1'].append(' '.join([item.strip() for item in df_Table[2][j].splitlines()]).strip())
                        sqldict['RegCtry'].append('UG')
                        sqldict['Cntry'].append('UG')
                        sqldict['Phone'].append(' '.join([item.strip() for item in df_Table[3][j].splitlines()]).replace('+','').replace('256','+256').strip())
                        sqldict['Fax'].append(' '.join([item.strip() for item in df_Table[4][j].splitlines()]).strip())
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append(d)
                        sqldict['RegCode'].append('BUG')
                        sqldict['RegulationType'].append('Supervised')
                        
                        # This part is susceptible to change due to the inconsistancy of the last column of the 1st and last tables
                        #print('-----------------------------')
                        my_list = [item.strip() for item in df_Table[6][j].splitlines() if item.strip() != '']
                        if len(my_list) > 1:
                            
                            if len(my_list[0]) != 1 and len(my_list) == 2:
                                #print('Website: ',my_list[1])
                                #print('Email: ',my_list[0])
                                sqldict['Website'].append(my_list[1])
                                sqldict['Email'].append(my_list[0])
                            
                            elif len(my_list[0]) == 1 and len(my_list[1]) != 1 and len(my_list) == 4:
                                if my_list[-1] != 'ug':
                                    #print('Website: ',my_list[1]+my_list[2])
                                    #print('Email: ',my_list[0]+my_list[-1])
                                    sqldict['Website'].append(my_list[1]+my_list[2])
                                    sqldict['Email'].append(my_list[0]+my_list[-1])
                                else:
                                    #print('Website: ',my_list[0]+my_list[2]+my_list[-1])
                                    #print('Email: ',my_list[1])
                                    sqldict['Website'].append(my_list[0]+my_list[2]+my_list[-1])
                                    sqldict['Email'].append(my_list[1])
                                
                            elif len(my_list[0]) == 1 and len(my_list[1]) != 1 and len(my_list) == 5:
                                #print('Website: ',my_list[0]+my_list[3]+my_list[-1])
                                #print('Email: ',my_list[1]+my_list[2])
                                sqldict['Website'].append(my_list[0]+my_list[3]+my_list[-1])
                                sqldict['Email'].append(my_list[1]+my_list[2])
                                
                            elif len(my_list[0]) == 1 and len(my_list[1]) == 1:
                                #print('Website: ',my_list[1]+my_list[-1])
                                #print('Email: ',my_list[0]+my_list[-2])
                                sqldict['Website'].append(my_list[1]+my_list[-1])
                                sqldict['Email'].append(my_list[0]+my_list[-2])
                                
                            elif len(my_list[0]) == 1 and len(my_list) == 2:
                                if len(findUrl(my_list[0]+my_list[1]))>0:
                                    #print('Website: ',findUrl(my_list[0]+my_list[1])[0])
                                    #print('Email: ','')
                                    sqldict['Website'].append(findUrl(my_list[0]+my_list[1])[0])
                                    sqldict['Email'].append('')
                                else:
                                    #print('Website: ','')
                                    #print('Email: ',my_list[0]+my_list[1])
                                    sqldict['Website'].append('')
                                    sqldict['Email'].append(my_list[0]+my_list[1])
                                    
                            elif len(my_list[0]) == 1 and len(my_list[1]) > 1 and len(my_list) == 3:
                                #print('Website: ',my_list[0]+my_list[-1])
                                #print('Email: ',my_list[1])
                                sqldict['Website'].append(my_list[0]+my_list[-1])
                                sqldict['Email'].append(my_list[1])
                                    
                        else:
                            #print('Website: ',my_list[0])
                            #print('Email: ','')
                            sqldict['Website'].append(my_list[0])
                            sqldict['Email'].append('')
                        #print('-----------------------------')
                        # Fill the rest with empty string
                        for key in sqldict.keys():
                            if len(sqldict['Name']) > len(sqldict[key]):
                                sqldict[key].append('')
                    
            os.remove(filePath)

        elif d in [3,4]:
            driver.find_element(By.XPATH, '//*[@id="apollo-page"]/section/div/div[2]/div[2]/div[1]/div/div/div[2]/div/div/div/ul[1]/li['+ str(d) +']/a').click()
            while len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:
                print('Waiting for file to download')
                sleep(2)
    
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
            
            #print(m,'---',typology[m].text)
            
            tables = tabula.read_pdf(filePath, pages="all")
            if type(tables) != list():
                tables = [tables]
            for i in range(len(tables)):
                df_Table = tables[i]
                colnames = [str(k) for k in range(len(df_Table.columns))]
                df_Table = pd.DataFrame(np.array(df_Table),columns=colnames)
                
                for j in range (len(df_Table)):
                    
                    if d == 3 and df_Table['0'][j] != 'NAME' and df_Table['0'][j] == df_Table['0'][j]:
                        #print('Name: ',df_Table['0'][j].strip())
                        #print('Address: ',df_Table['1'][j].strip())
                        #print('TEL: ',df_Table['2'][j])
                        #print('EMAIL: ',df_Table['3'][j],'\n')
                        sqldict['Name'].append(df_Table['0'][j].strip())
                        sqldict['Address_1'].append(df_Table['1'][j].strip())
                        sqldict['Email'].append(df_Table['3'][j])
                        sqldict['RegCtry'].append('UG')
                        sqldict['Cntry'].append('UG')
                        sqldict['Phone'].append(df_Table['2'][j])
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append(d)
                        sqldict['RegCode'].append('BUG')
                        sqldict['RegulationType'].append('Supervised')
                        
                    elif d == 4 and df_Table['1'][j] != 'NAME' and df_Table['0'][j]==df_Table['0'][j]:
                        if len(pattern.findall(df_Table['0'][j])) > 0:
                            #print(df_Table['1'][j],'\n')
                            #print('Name: ',df_Table['1'][j].strip())
                            #print('Address: ',df_Table['2'][j].strip())
                            #print('TEL: ',df_Table['3'][j])
                            #print('EMAIL: ',df_Table['4'][j],'\n')
                            sqldict['Name'].append(df_Table['1'][j].strip())
                            sqldict['Address_1'].append(df_Table['2'][j].strip())
                            sqldict['Email'].append(df_Table['4'][j])
                            sqldict['RegCtry'].append('UG')
                            sqldict['Cntry'].append('UG')
                            sqldict['Phone'].append(df_Table['3'][j])
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['ListCode'].append(d)
                            sqldict['RegCode'].append('BUG')
                            sqldict['RegulationType'].append('Supervised')
                        elif df_Table['0'][j] != 'MDIs offering Money Remittance Services':
                            #print(df_Table['0'][j],'\n')
                            #print('Name: ',df_Table['0'][j].strip())
                            #print('Address: ',df_Table['1'][j].strip())
                            #print('TEL: ',df_Table['2'][j])
                            #print('EMAIL: ',df_Table['3'][j],'\n')
                            sqldict['Name'].append(df_Table['0'][j].strip())
                            sqldict['Address_1'].append(df_Table['1'][j].strip())
                            sqldict['Email'].append(df_Table['3'][j])
                            sqldict['RegCtry'].append('UG')
                            sqldict['Cntry'].append('UG')
                            sqldict['Phone'].append(df_Table['2'][j])
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['ListCode'].append(d)
                            sqldict['RegCode'].append('BUG')
                            sqldict['RegulationType'].append('Supervised')

                    # Fill the rest with empty string
                    for key in sqldict.keys():
                        if len(sqldict['Name']) > len(sqldict[key]):
                            sqldict[key].append('')
                            
            os.remove(filePath)
                    
        else: 
            pass
        
df=pd.DataFrame(sqldict)
writer = ExcelWriter(filename)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()

sleep(3)

driver.quit()
    
    